# 01D — Fresh Re-audit of Leakage-Controlled Split V2

**NO TRAINING.**

Run after `01C_Rebuild_With_01B_Pairs.ipynb`.

This repeats the exact-hash + pHash + SIFT/RANSAC scan from scratch on:
`Cataract/Data_Clean_LeakageControlled_v2`

Training is allowed only if the final output says:

`PASS ✅ V2 CLEAN SPLIT VERIFIED.`

In [1]:
import sys,subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless'],check=True)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, cv2, numpy as np, pandas as pd

PROJECT=Path('/content/drive/MyDrive/Cataract')
ROOT=PROJECT/'Data_Clean_LeakageControlled_v2'
OUT=PROJECT/'FINAL_REVISION_2026_08'/'clean_split_audit_v2'/'fresh_reaudit'
OUT.mkdir(parents=True,exist_ok=True)

CLASS_ORDER=['Cataract','Normal','Not Eye']
SPLITS=['Train','Validation','Test']
assert ROOT.exists(),f'Run 01C first: {ROOT}'

Mounted at /content/drive


In [2]:
def pack64(bits):
    return np.packbits(bits.astype(np.uint8).reshape(-1)).tobytes().hex()

def phash(gray):
    small=cv2.resize(gray,(32,32),interpolation=cv2.INTER_AREA).astype(np.float32)
    D=cv2.dct(small)[:8,:8]
    med=np.median(D.flatten()[1:])
    return pack64(D.flatten()>med)

def fingerprint(p):
    data=p.read_bytes()
    sha=hashlib.sha256(data).hexdigest()
    img=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)
    if img is None:return None
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    rel=p.relative_to(ROOT).as_posix()
    split,cls,fn=rel.split('/',2)
    return {'path':rel,'split':split,'class':cls,'filename':fn,
            'sha256':sha,'phash64':phash(gray)}

paths=[p for s in SPLITS for c in CLASS_ORDER for p in (ROOT/s/c).glob('*') if p.is_file()]
print('Images:',len(paths))

rows=[]
for i,p in enumerate(paths,1):
    r=fingerprint(p)
    if r:rows.append(r)
    if i%1000==0:print(i,'/',len(paths))

df=pd.DataFrame(rows)
df.to_csv(OUT/'fingerprints_v2.csv',index=False)
print(df.groupby(['split','class']).size())

Images: 13624
1000 / 13624
2000 / 13624
3000 / 13624
4000 / 13624
5000 / 13624
6000 / 13624
7000 / 13624
8000 / 13624
9000 / 13624
10000 / 13624
11000 / 13624
12000 / 13624
13000 / 13624
split       class   
Test        Cataract     855
            Normal       978
            Not Eye      756
Train       Cataract    2926
            Normal      3344
            Not Eye     2584
Validation  Cataract     721
            Normal       824
            Not Eye      636
dtype: int64


In [3]:
bad=[]
for sha,g in df.groupby('sha256'):
    if g.split.nunique()>1:bad.append(g)
cross_exact=pd.concat(bad,ignore_index=True) if bad else pd.DataFrame(columns=df.columns)
cross_exact.to_csv(OUT/'cross_partition_exact_duplicates_v2.csv',index=False)
print('Exact cross-partition duplicate rows:',len(cross_exact))

Exact cross-partition duplicate rows: 0


In [4]:
LUT=np.array([bin(i).count('1') for i in range(256)],dtype=np.uint8)
def to_u64(x):return np.uint64(int(x,16))
def hdist(arr,x):
    v=np.bitwise_xor(arr,np.uint64(x))
    return LUT[v.view(np.uint8).reshape(-1,8)].sum(axis=1)

df['ph_u']=df.phash64.map(to_u64)
comparisons=[('Validation','Train'),('Test','Train'),('Test','Validation')]
cand=[]

for qsplit,rsplit in comparisons:
    q=df[df.split==qsplit].reset_index(drop=True)
    r=df[df.split==rsplit].reset_index(drop=True)
    rph=r.ph_u.to_numpy(np.uint64)

    for _,x in q.iterrows():
        d=hdist(rph,x.ph_u)
        same=(r['class'].to_numpy()==x['class'])
        if same.any():
            ids=np.where(same)[0]
            j=ids[np.argmin(d[same])]
            cand.append({
                'query_path':x.path,'query_split':qsplit,'query_class':x['class'],
                'ref_path':r.iloc[j].path,'ref_split':rsplit,'ref_class':r.iloc[j]['class'],
                'phash_distance':int(d[j]),'candidate_type':'same_class'
            })

        j=int(np.argmin(d))
        cand.append({
            'query_path':x.path,'query_split':qsplit,'query_class':x['class'],
            'ref_path':r.iloc[j].path,'ref_split':rsplit,'ref_class':r.iloc[j]['class'],
            'phash_distance':int(d[j]),'candidate_type':'nearest_any'
        })

cands=pd.DataFrame(cand)
cands.to_csv(OUT/'phash_candidates_v2.csv',index=False)
print('Candidates with pHash <=12:',int((cands.phash_distance<=12).sum()))

Candidates with pHash <=12: 1586


In [5]:
susp=cands[cands.phash_distance<=12].drop_duplicates(['query_path','ref_path']).copy()

def prep(rel):
    im=cv2.imread(str(ROOT/rel),cv2.IMREAD_GRAYSCALE)
    h,w=im.shape
    sc=min(1.0,512/max(h,w))
    if sc<1:
        im=cv2.resize(im,(int(w*sc),int(h*sc)),interpolation=cv2.INTER_AREA)
    return im

def verify(rec):
    a,b=prep(rec['query_path']),prep(rec['ref_path'])
    sift=cv2.SIFT_create(nfeatures=1200,contrastThreshold=0.02)
    k1,x1=sift.detectAndCompute(a,None)
    k2,x2=sift.detectAndCompute(b,None)
    good=[];inl=0;ratio=0.0

    if x1 is not None and x2 is not None and len(x1)>=2 and len(x2)>=2:
        bf=cv2.BFMatcher(cv2.NORM_L2)
        matches=bf.knnMatch(x1,x2,k=2)
        good=[p for p,q in matches if p.distance<0.75*q.distance]
        if len(good)>=4:
            src=np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
            dst=np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
            H,mask=cv2.findHomography(src,dst,cv2.RANSAC,5.0)
            if mask is not None:
                inl=int(mask.sum());ratio=inl/len(good)

    out=dict(rec)
    out.update({
        'sift_good':len(good),
        'sift_inliers':inl,
        'sift_inlier_ratio':ratio,
        'confirmed_strict':bool(len(good)>=20 and inl>=15 and ratio>=0.5)
    })
    return out

verified=[]
for i,r in enumerate(susp.to_dict('records'),1):
    verified.append(verify(r))
    if i%100==0:print(i,'/',len(susp))

ver=pd.DataFrame(verified)
ver.to_csv(OUT/'sift_verified_v2.csv',index=False)

confirmed=ver[ver.confirmed_strict] if len(ver) else ver
confirmed.to_csv(OUT/'CONFIRMED_CROSS_PARTITION_NEAR_DUPLICATES_v2.csv',index=False)

print('\nConfirmed strict cross-partition near-duplicate pairs:',len(confirmed))

if len(cross_exact)==0 and len(confirmed)==0:
    print('\nPASS ✅ V2 CLEAN SPLIT VERIFIED. You may begin model training.')
else:
    print('\nSTOP ❌ Do not train yet. Another family-expansion pass is required.')

100 / 903
200 / 903
300 / 903
400 / 903
500 / 903
600 / 903
700 / 903
800 / 903
900 / 903

Confirmed strict cross-partition near-duplicate pairs: 281

STOP ❌ Do not train yet. Another family-expansion pass is required.
